# **XGBoost** классификация


это реализация градиентного бустинга над деревьями решений с регуляризацией, которая последовательно добавляет новые деревья, уменьшая ошибку предыдущих и тем самым строя сильную модель из множества слабых.


## Предобработка

In [7]:
md = "XGBoost"
st = "train_test"
name = "AFKS"

In [8]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [9]:
import pandas as pd
from data_preprocessing.DataForModel import prep, append_results
from metrics.Metrics import ttp_metrics

In [10]:
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

## Обучение

In [11]:
from xgboost import XGBClassifier

In [14]:
def merge_ttp_classes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge TTP classes:
    0,1 -> good_fast_mid
    2   -> slow
    3   -> no_profit
    """

    df = df.copy()

    df["TTP_merged"] = df["TTP_class"].map({
        0: 0,  # good_fast_mid
        1: 0,  # good_fast_mid
        2: 1,  # slow
        3: 2   # no_profit
    })

    return df

In [15]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0,1,2,3])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train.values)
class_weight = dict(zip(classes, cw))

sample_weight = y_train.map(class_weight).values
model.fit(X_train, y_train, sample_weight=sample_weight)

NameError: name 'y_train' is not defined

In [ ]:
def make_xgb_model_3class():
    return XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    )

In [ ]:
results_all = []

for fold, (X_train, X_test, y_train, y_test, scaler) in enumerate(
    prep(
        df=df,
        horizons=[12, 24, 48],
        target_col="TTP_class",
        train_size=600,
        test_size=100,
        step=100
    )
):
    # --- MERGE TARGET ---
    y_train_m = y_train.map({0:0, 1:0, 2:1, 3:2})
    y_test_m  = y_test.map({0:0, 1:0, 2:1, 3:2})

    model = make_xgb_model_3class()

    model.fit(X_train, y_train_m)

    y_pred = model.predict(X_test)

    metrics = compute_merged_metrics(
        y_true=y_test_m.values,
        y_pred=y_pred
    )

    result_row = {
        "model": "XGB_TTP_MERGED",
        "fold": fold,
        "recall_good": metrics["recall_good"],
        "precision_no_profit": metrics["precision_no_profit"]
    }

    append_results(result_row)

    print(f"\nFold {fold}")
    print(result_row)
    print("Confusion matrix:\n", metrics["confusion_matrix"])

Results appended to /Users/side/Desktop/Trading Chaos AI/df/results/Results.csv

Fold 0
{'model': 'XGB_TTP_MERGED', 'fold': 0, 'recall_good': np.float64(0.873015873015873), 'precision_no_profit': np.float64(0.6363636363636364)}
Confusion matrix:
 [[55  0  8]
 [ 5  0  4]
 [ 7  0 21]]
Results appended to /Users/side/Desktop/Trading Chaos AI/df/results/Results.csv

Fold 1
{'model': 'XGB_TTP_MERGED', 'fold': 1, 'recall_good': np.float64(0.8888888888888888), 'precision_no_profit': np.float64(0.696969696969697)}
Confusion matrix:
 [[56  0  7]
 [ 1  0  3]
 [10  0 23]]
Results appended to /Users/side/Desktop/Trading Chaos AI/df/results/Results.csv

Fold 2
{'model': 'XGB_TTP_MERGED', 'fold': 2, 'recall_good': np.float64(0.875), 'precision_no_profit': np.float64(0.6333333333333333)}
Confusion matrix:
 [[56  1  7]
 [ 4  0  4]
 [ 8  1 19]]
Results appended to /Users/side/Desktop/Trading Chaos AI/df/results/Results.csv

Fold 3
{'model': 'XGB_TTP_MERGED', 'fold': 3, 'recall_good': np.float64(0.94736

In [ ]:
results = {
    "model": md,
    "target_type": tr,
    "accuracy": accuracy
}

results_df = pd.DataFrame([results])
results_df.to_csv(f"/Users/side/Desktop/Trading Chaos AI/df/results/Results.csv", index=False)

настройка баланса классов

In [ ]:
results = {
    "model": "XGBoost",
    "task": "classification",
    "horizon": 20,
    "accuracy": accuracy
}

append_results(results)

Results appended to /Users/side/Desktop/Trading Chaos AI/df/results/Results.csv
